# 03. Learn a normal-motion reference without collapse

This lesson follows one tiny normal-only training run from visible tokens through teacher targets and the loss. The implementation is a compact Skeleton-JEPA adaptation with a coordinate readout and VICReg-style regularization, not a numerical reproduction of the original action-recognition paper.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Follow the three networks

The online view encoder sees visible joint-time tokens. The predictor combines their embeddings with positioned mask tokens and predicts hidden representations. The target encoder sees the complete observed sequence and produces stop-gradient targets. Its weights follow the online encoder through an exponential moving average.

This teaching model uses one joint-time token and final-layer teacher features. Published Skeleton-JEPA uses a different tokenization and target construction. We preserve the conceptual learning path while declaring the adaptation.

In [ ]:
display(SVG(filename=str(PROJECT / "images/jepa.svg")))

In [ ]:
from worldmodels.notebook_support import lesson_data
from worldmodels.model import ModelConfig, SkeletonJEPA, loss_terms, variance_covariance
from worldmodels.masking import uniform_mask
from worldmodels.training import TrainConfig, fit, encode
fixture = lesson_data(n=16, seed=27)
model = SkeletonJEPA(ModelConfig(dim=24, encoder_layers=1))
print("Normal-only labels:", np.unique(fixture["labels"]))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## Step 2. Inspect one prediction before training

The teacher is not an independently supervised clinician. It is a slowly updated copy of the online encoder. Its output becomes a centered and sharpened categorical distribution across latent dimensions. The predictor distribution is trained against that target using cross-entropy at masked positions. A separate coordinate head makes approximate skeleton trajectories visible; latent prediction alone does not automatically reconstruct coordinates.

In [ ]:
bx = torch.as_tensor(fixture["x"][:4]); bv = torch.as_tensor(fixture["valid"][:4])
target = uniform_mask(bv, generator=torch.Generator().manual_seed(27))
model.train()
output = model(bx, bv, target)
terms = loss_terms(model, output, bx, bv, target)
display(pd.Series({k: float(v.detach()) for k,v in terms.items()}, name="Before training"))
assert all(not p.requires_grad for p in model.teacher.parameters())

## Step 3. Explain collapse and the actual role of VICReg

If every input becomes the same embedding, prediction can become uninformative even when a loss appears stable. The variance term penalizes dimensions with too little batch variation. The covariance term reduces redundant dimensions. Together with centered teacher targets and EMA, these are explicit anti-collapse mechanisms.

VICReg does not push diagnosis clusters as far apart as possible. It uses no class labels here. Whether useful clinical structure appears is an empirical question. Batches should contain distinct sources, because many nearly identical windows from one recording do not provide independent variation.

In [ ]:
spread = output["representation"].detach() if REAL else torch.randn(8,24)
collapsed = torch.zeros_like(spread)
comparison = {name: [float(v) for v in variance_covariance(z)] for name,z in [("collapsed", collapsed),("observed representation" if REAL else "random spread", spread)]}
display(pd.DataFrame(comparison, index=["variance penalty", "covariance penalty"]))

## Step 4. Train only on normal training sources

In real mode the lesson loads normal training windows with actual source and sequence IDs. In synthetic mode the fixture labels are normal and its source IDs are explicitly artificial. The API refuses non-normal labels in the normal-reference regime and refuses any partition value other than `train`. Four updates are enough to exercise the mechanics; they are not convergence evidence. The full experiment uses actual source IDs and separately recorded update budgets.

In [ ]:
history = fit(model, fixture["x"], fixture["valid"],
              TrainConfig(steps=4, batch_size=8, seed=27, regime="normal", run_id="lesson03-normal-v1"),
              labels=fixture["labels"], source_ids=fixture["source_ids"], partitions=["train"]*len(fixture["x"]))
display(pd.DataFrame(history))

## Step 5. Check representations rather than only loss

Look at variation across examples, dimensions, and sources. An embedding with zero variation is unusable even if training finishes. A nonzero standard deviation is only a diagnostic, not proof of useful structure. Clinical separation and robustness must be tested on held-out sources with suitable baselines.

In [ ]:
embeddings = encode(model, fixture["x"], fixture["valid"])
print("Embedding matrix:", embeddings.shape)
print("Median dimension standard deviation:", float(np.median(embeddings.std(0))))
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(pd.DataFrame(history)["step"], pd.DataFrame(history)["loss"], marker="o")
ax.set(xlabel="Update", ylabel="Training objective", title=EVIDENCE_LABEL + " normal-only training exercise")
plt.show()

## Step 6. Preserve the reference identity

Adaptation should start from a copy of this normal checkpoint. Replacing it in place would destroy the reference needed to ask whether a new gait deviates from learned normal motion. Checkpoint metadata includes sources, data identity, code hashes, training settings, and parent identity. New training stages require new run names.

In [ ]:
print({k: model.run_metadata[k] for k in ("run_id", "regime", "n_windows", "data_sha256", "parent_run_id")})
print("Teacher remains gradient-free:", all(p.grad is None for p in model.teacher.parameters()))

In [ ]:
lesson_evidence = persist_lesson("03_normal_only_jepa_and_vicreg", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.